# 25. Adaptive Retrieval

Classify the question first, then choose a retrieval strategy that fits it.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

One strategy doesn't fit every question. *'What dimension are the vectors?'* needs one precise chunk; *'Compare dense and hybrid retrieval'* needs broad coverage; *'What is 12 x 12?'* needs no retrieval at all. **Adaptive retrieval** classifies the query first and routes it to the right strategy.

## How it works

1. Classify the query: factual / analytical / opinion / no-retrieval
2. Factual: small k, precise
3. Analytical: decompose into sub-questions, retrieve for each
4. Opinion: wide retrieval for diverse viewpoints
5. No-retrieval: answer directly with the LLM

```text
 question --> [classifier] --+--> factual    -> k=2
                             +--> analytical -> sub-queries -> merge
                             +--> opinion    -> k=6 diverse
                             +--> none       -> LLM only
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 10:55:31.771000 8233 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x7a68040000>
    label = <none> 
    device = <AGXG13GDevice: 0x7a5cf14000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7a67fb8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7a5cf14000>
            name = Apple M1 
    retainedReferences = 1


In [2]:
STRATEGIES = {
    "factual":    "look up one specific fact",
    "analytical": "needs broad, multi-part coverage",
    "opinion":    "asks for recommendations or trade-offs",
    "no_retrieval": "general chat or maths that needs no documents",
}

def classify(query):
    options = "\n".join(f"- {k}: {v}" for k, v in STRATEGIES.items())
    label = llm(f"Classify the question into exactly one category. Reply with the category name only.\n{options}\n\nQuestion: {query}").lower()
    return next((k for k in STRATEGIES if k in label), "factual")

In [3]:
def adaptive_rag(query):
    kind = classify(query)
    if kind == "no_retrieval":
        return kind, llm(query)
    if kind == "factual":
        contexts = [h["text"] for h in store.search(query, 2)]
    elif kind == "analytical":
        subs = llm_json(f"Split into 3 sub-questions. JSON list of strings.\n\n{query}") or [query]
        contexts = list({h["text"] for s in subs for h in store.search(s, 2)})
    else:  # opinion: gather diverse viewpoints with a wider net
        contexts = [h["text"] for h in store.search(query, 6)]
    return kind, answer(query, contexts)

for q in ["What dimension are all-MiniLM-L6-v2 vectors?", "Compare dense, BM25 and hybrid retrieval and how reranking fits in", "Should I fine-tune or use RAG?", "What is 12 times 12?"]:
    kind, response = adaptive_rag(q)
    print(f"[{kind}] {q}\n  {response[:300]}\n")

[factual] What dimension are all-MiniLM-L6-v2 vectors?
  I don't know. The provided context does not mention the dimension of MiniLM-L6-v2 vectors. It only discusses vector search indexes and chunking strategies for text indexing.



[analytical] Compare dense, BM25 and hybrid retrieval and how reranking fits in
  I don't know. The provided context doesn't mention anything about comparison between dense, BM25, and hybrid retrieval or how reranking fits into them.



Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x7a68040700>
    label = <none> 
    device = <AGXG13GDevice: 0x7a5cf14000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7a67fb8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7a5cf14000>
            name = Apple M1 
    retainedReferences = 1


[opinion] Should I fine-tune or use RAG?
  According to [4], RAG is best for facts that change or must be traceable to a source.



Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x7a68040380>
    label = <none> 
    device = <AGXG13GDevice: 0x7a5cf14000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7a67fb8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7a5cf14000>
            name = Apple M1 
    retainedReferences = 1


[factual] What is 12 times 12?
  I don't know. The provided context doesn't mention mathematical calculations or operations like multiplication. It discusses the differences between fine-tuning a model, RAG systems, caching, and latency, but doesn't provide information on how to perform specific calculations.



## How accurate is the router?
The router only helps if it classifies correctly. Test it on labelled examples.

In [4]:
import pandas as pd
labelled = [
    ("What is the cosine similarity range?", "factual"),
    ("How many chunks should I retrieve?", "factual"),
    ("Compare BM25, dense retrieval and hybrid search", "analytical"),
    ("Explain how agentic RAG differs from classic RAG and why it matters", "analytical"),
    ("Should I fine-tune or use RAG for my product?", "opinion"),
    ("Which vector database would you recommend?", "opinion"),
    ("What is 15 plus 27?", "no_retrieval"),
    ("Write a haiku about spring", "no_retrieval"),
]
rows = [{"question": q, "expected": e, "predicted": classify(q)} for q, e in labelled]
rt = pd.DataFrame(rows); rt["ok"] = rt.expected == rt.predicted
display(rt); print("router accuracy:", rt.ok.mean())

,question,expected,predicted,ok
0,What is the cosine similarity range?,factual,factual,True
1,How many chunks should I retrieve?,factual,no_retrieval,False
2,"Compare BM25, dense retrieval and hybrid search",analytical,analytical,True
3,Explain how agentic RAG differs from classic R...,analytical,analytical,True
4,Should I fine-tune or use RAG for my product?,opinion,opinion,True
5,Which vector database would you recommend?,opinion,opinion,True
6,What is 15 plus 27?,no_retrieval,no_retrieval,True
7,Write a haiku about spring,no_retrieval,opinion,False


router accuracy: 0.75


## Does adapting help answers?

In [5]:
compare_pipelines({"always k=3": lambda q: rag(q, store, 3)[0], "adaptive": lambda q: adaptive_rag(q)[1]}, n=5)

Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x7a68040380>
    label = <none> 
    device = <AGXG13GDevice: 0x7a5cf14000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7a67fb8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7a5cf14000>
            name = Apple M1 
    retainedReferences = 1


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13GFamilyCommandBuffer: 0x7a68040a80>
    label = <none> 
    device = <AGXG13GDevice: 0x7a5cf14000>
        name = Apple M1 
    commandQueue = <AGXG13GFamilyCommandQueue: 0x7a67fb8000>
        label = <none> 
        device = <AGXG13GDevice: 0x7a5cf14000>
            name = Apple M1 
    retainedReferences = 1


,keyword_recall,judge_correct,avg_seconds
pipeline,,,
always k=3,0.65,0.4,11.9
adaptive,0.33,0.2,11.4


## Latency by route
Cheap routes should be fast.

In [6]:
import time
for q in ["What dimension are all-MiniLM-L6-v2 vectors?", "Compare dense and hybrid retrieval", "What is 2 plus 2?"]:
    t = time.time(); kind, _ = adaptive_rag(q); print(f"{kind:13s} {time.time() - t:5.1f}s  {q}")

factual         2.1s  What dimension are all-MiniLM-L6-v2 vectors?


analytical     30.0s  Compare dense and hybrid retrieval


no_retrieval    2.3s  What is 2 plus 2?


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Right effort for each question | Router errors send questions down the wrong path |
| Skips retrieval when unnecessary (saves cost) | Extra LLM call to classify |
| Easy to add new routes | More code paths to test |

## When to use it

Mixed workloads: chat + lookup + analysis in one assistant.

## Experiments to try

- Train a small classifier (logistic regression on embeddings) instead of using the LLM.
- Add a 'needs a calculator/SQL' route.
- Log the route taken for analytics.

## Key takeaways

- Classify, then choose a strategy.
- Measure the router itself.
- A cheap router can be a tiny embedding classifier.

---
**Next:** [26. Explainable Retrieval](../26_explainable_retrieval/explainable_retrieval.ipynb)